In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import matplotlib.pyplot as plt

In [4]:
def generate_temperature_data(seq_length, num_samples):
    np.random.seed(42)
    base_temp = 20      # Average temperature in degrees Celsius
    temp_variation = 5  # Maximum daily temperature variation

    data = base_temp + temp_variation * np.sin(np.linspace(0, 100, num_samples)) + np.random.normal(0, 1, num_samples)

    X = []
    y = []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])

    return np.array(X), np.array(y)

In [5]:
seq_length = 30
num_samples = 365  # Simulating a year of data
X, y = generate_temperature_data(seq_length, num_samples)
X = X[..., np.newaxis]  # Add an extra dimension for compatibility with RNN input

In [6]:
model = Sequential([
    SimpleRNN(50, activation='tanh', input_shape=(seq_length, 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()


C:\Users\PT\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)               │ (None, 50)                  │           2,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,651 (10.36 KB)

 Trainable params: 2,651 (10.36 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - loss: 404.5452 - val_loss: 400.7564
Epoch 2/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 363.4200 - val_loss: 361.5700
Epoch 3/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 327.4482 - val_loss: 328.0181
Epoch 4/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 296.1010 - val_loss: 297.0657
Epoch 5/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 265.8916 - val_loss: 266.2329
Epoch 6/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 235.7998 - val_loss: 235.5552
Epoch 7/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 206.3658 - val_loss: 206.3751
Epoch 8/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 180.2146 - val_loss: 181.9835
Epoch 9/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 158.6879 - val_loss: 161.2748
Epoch 10/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 140.9699 - val_loss: 145.0960
Epoch 11/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 127.3446 - val_loss: 132.6395
Epoch 12/20
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 1

In [8]:
def predict_future(model, recent_data, days):
    predictions = []
    input_seq = recent_data[-seq_length:]

    for _ in range(days):
        pred = model.predict(input_seq[np.newaxis, ...])[0, 0]
        predictions.append(pred)
        input_seq = np.append(input_seq[1:], [[pred]], axis=0)

    return predictions

In [9]:
future_predictions = predict_future(model, X[-1], 7)
print("Predicted Temperatures for Next 7 Days:", future_predictions)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
Predicted Temperatures for Next 7 Days: [np.float32(12.649191), np.float32(12.634717), np.float32(12.633974), np.float32(12.633956), np.float32(12.633957), np.float32(12.633956), np.float32(12.633956)]
